# Imports

In [2]:
import time
import numpy as np
from typing import Callable

# Data Loading

## Random Sampled Data

In [5]:
rng = np.random.default_rng(seed=42)
m = 1000
d = 1000
k = 50

A = rng.random((m, d))
X = rng.random((d, k))
E = rng.random((m, k))


scores = np.dot(A, X) + E
b = np.argmax(scores, axis = 1)

## Dataset Loading

# Method Calls
This section is for implementing all the methods that will be used for generating graphs or any other types of solutions

In [ ]:
class MultiLogisticRegressionOracle:
    def __init__(self, predictors: np.array, predictions: list, n_categories: int):
        self.predictors = predictors # Referenced as A
        self.predictions = predictions # Referenced as b

        self.n_features = predictors.shape[1] # Referenced as d
        self.n_categories = n_categories # Referenced as k

        self.indicator = np.array([predictions == c for c in range(self.n_categories)]).T
        self.indicator = self.indicator.astype(int) # m x k


    def loss(self, x: np.array):
        scores = np.dot(self.predictors, x) # m x k
        exp = np.exp(scores)
        sum_exp = np.sum(exp, axis=1)
        log_sum_exp = np.log(sum_exp)

        class_scores = scores[range(len(self.predictions)), self.predictions]
        result = -np.sum(class_scores) + np.sum(log_sum_exp)
        return result


    def __call__(self, x: np.array, j_block: int|None = None):
        """Estimate the gradient in regard of specific categories."""
        transformed_entries = np.exp(np.dot(self.predictors, x)) # m x k
        summed_entries = np.sum(transformed_entries, axis=1, keepdims=True)
        softmax_entries = transformed_entries / summed_entries # m x k

        if j_block is not None:
            result = np.zeros((self.n_features, self.n_categories)) # d x k

            result[j_block] = -np.dot(self.predictors.T[j_block,:], self.indicator - softmax_entries)
        else:
            result = -np.dot(self.predictors.T, self.indicator - softmax_entries)

        return result

class CachedMultiLogisticRegressionOracle(MultiLogisticRegressionOracle):
    def __init__(self, predictors, predictions, n_categories):
        super().__init__(predictors, predictions, n_categories)
        self.S = np.zeros((self.predictors.shape[0], self.n_categories)) #Initializing scores AX (m x k)

    def update_cache(self, i_star, delta_x_row):
        self.S += np.outer(self.predictors[:, i_star], delta_x_row)

    def __call__(self):
        exp_S = np.exp(self.S)
        summed_entries = np.sum(exp_S, axis=1, keepdims=True)
        softmax_entries = exp_S / summed_entries

        result = -np.dot(self.predictors.T, self.indicator - softmax_entries)

        return result

def accuracy(predictors: np.array, ground_truth:np.array, regresssors: np.array) -> float:
    predictions = np.argmax(np.dot(predictors, regresssors), axis=1)
    return np.sum(predictions == ground_truth)/len(ground_truth)

In [10]:
# Block step size
def block_step_size(A):

    column_norms_sq = np.sum(A**2, axis=0)
    block_L = 0.25 * column_norms_sq
    block_step_size = 1.0 / block_L

    return block_step_size

In [ ]:
# Gradient Descent
def gradient_descent(
    x_init: np.array,
    oracle: Callable,
    alpha: float,
    tol: float = 1e-6,
    debug_logs: bool = False,
    max_iter: int = 5000
    ) -> tuple[np.array, int]:

    x_k = x_init
    for k in range(1, max_iter + 1):
        gradient = oracle(x_k)
        gradient_norm = np.linalg.norm(gradient)

        if k == 1000000:# gradient_norm < tol:
            print(f"Converged at iteration {k}. Gradient Norm: {gradient_norm}")
            break
        x_k_1 = x_k - alpha * gradient

        if debug_logs == True:
            if k % 500 == 0:
                acc = accuracy(oracle.predictors, oracle.predictions, x_k_1)
                print(f"iteration: {k}, selected_block: {i_star}, sequential distance: {np.linalg.norm(x_k - x_k_1)}, loss: {oracle.loss(x_k_1):.6f}")
                print(f"accuracy: {acc*100:.2f}%")

        if k % 5 == 0:
            current_loss = oracle.loss(x_k_1)
            current_acc = accuracy(oracle.predictors, oracle.predictions, x_k_1)
            print(f"Iteration {k}: Loss = {current_loss}, Accuracy = {current_acc}")

        x_k = x_k_1

    return (x_k, k)

In [4]:
# GS BCGD
def gs_bcgd(x_init: np.array,
            oracle: Callable,
            block_step_size: np.array,
            max_iter: int = 5000,
            debug_logs: bool = False,
            tol: float = 1e-6
            ) -> np.array:

    x_k = x_init.copy()
    start_time = time.time()

    for k in range(1, max_iter + 1):
        #GS selection rule
        full_grad = oracle() #gets the gradient for the whole X
        block_norms = np.linalg.norm(full_grad, axis=1) #calculates the norm of each block (row)

        i_star = np.argmax(block_norms) #selects the block with the highest gradient norm

        #Update
        x_k_1 = x_k.copy()

        delta_x_row = -block_step_size[i_star]*full_grad[i_star, :]

        x_k_1[i_star, :] = x_k[i_star, :] + delta_x_row

        oracle.update_cache(i_star, delta_x_row)

        if debug_logs == True:
            if k % 500 == 0:
                acc = accuracy(oracle.predictors, oracle.predictions, x_k_1)
                print(f"iteration: {k}, selected_block: {i_star}, sequential distance: {np.linalg.norm(x_k - x_k_1)}, loss: {oracle.loss(x_k_1):.6f}")
                print(f"accuracy: {acc*100:.2f}%")

        last_dist = np.linalg.norm(x_k - x_k_1)

        if last_dist < tol:
          end_time = time.time()
          elapsed_time = end_time - start_time
          print(f"Duration: {elapsed_time:.2f} s")
          print(f"Converged at iteration {k}. Selected Block:{i_star}. Gradient Norm: {last_dist}. Loss:{oracle.loss(x_k):.4f}")
          break


        x_k = x_k_1

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Duration: {elapsed_time:.2f} s")
    print(f"\nFinal iteration: {max_iter+1}. Selected Block: {i_star}. Gradient Norm: {last_dist}. Loss {oracle.loss(x_k):.4f}")
    print(f"accuracy: {accuracy(oracle.predictors, oracle.predictions, x_k)*100:.2f}%\n")

    return x_k

In [ ]:
# Random BCGD
def randomized_bcgd(
    x_init: np.array,
    n_categories: int,
    oracle: Callable,
    alpha: np.array
) -> np.array:
    """Estimate BCGD optimization with Randomized Rule."""
    k = 1
    x_k = x_init

    while True:
        i_k = rng.integers(0, n_categories, 1)[0]

        g_i_k = oracle(x_k, i_k)
        x_k_1 = x_k - alpha * g_i_k

        if k >= 100000: # TODO: Define stop condition
            break

        if k%500 == 0: # Just for debugging purposes
            print(f"iteration: {k}, sequential distance: {np.linalg.norm(x_k-x_k_1)}, loss: {oracle.loss(x_k_1)}")
            print(f"accuracy: {accuracy(oracle.predictors, oracle.predictions, x_k_1)}")

        x_k = x_k_1
        k += 1

    return (x_k, k)

#### Gradient Descent

In [ ]:
x_init = np.zeros((d, k))
oracle = MultiLogisticRegressionOracle(predictors=A, predictions=b, n_categories=k)

#step size
spectral_norm = np.linalg.norm(A, ord=2)
L = (1 / (4 * m)) * (spectral_norm**2)
safe_alpha = (1 / L)/100

final_weights, total_steps = gradient_descent(
    x_init=x_init,
    oracle=oracle,
    alpha=safe_alpha)

In [22]:
np.linalg.norm(A, ord=2)

np.float64(500.18372273602455)

In [ ]:
# Block step size
def block_step_size(A):

    column_norms_sq = np.sum(A**2, axis=0)
    block_L = 0.25 * column_norms_sq
    block_step_size = 1.0 / block_L

    return block_step_size

In [26]:
spectral_norm = np.linalg.norm(A, ord=2)
L = (1 / (4 * m)) * (spectral_norm**2)
safe_alpha = (1 / L)/100


In [20]:
spectral_norm

np.float64(500.18372273602455)

In [27]:
safe_alpha

np.float64(0.00015988248222497173)

In [28]:
np.mean(block_step_size(A))

np.float64(0.012009266942004777)

In [29]:
block_step_size(A)

array([0.01203261, 0.01165373, 0.01223015, 0.01289164, 0.01192697,
       0.01192222, 0.01193586, 0.01167788, 0.01253027, 0.01200109,
       0.01153076, 0.0115454 , 0.01245208, 0.01229079, 0.0113985 ,
       0.01160449, 0.01191417, 0.01168482, 0.01190575, 0.01226667,
       0.01203619, 0.01224936, 0.01224981, 0.01180856, 0.01238351,
       0.01178413, 0.01163847, 0.01158964, 0.01211284, 0.01219112,
       0.0118965 , 0.01173259, 0.01237852, 0.01134787, 0.01226799,
       0.01226062, 0.01153648, 0.0117215 , 0.01166161, 0.01124043,
       0.01204845, 0.01170282, 0.01201874, 0.01193844, 0.0114029 ,
       0.01200929, 0.01227156, 0.01194266, 0.01211227, 0.01207982,
       0.0126462 , 0.01216156, 0.01226426, 0.0120404 , 0.01160944,
       0.01283627, 0.01185455, 0.01202842, 0.01151834, 0.01181045,
       0.01176715, 0.01163548, 0.01225776, 0.01233063, 0.01187114,
       0.01195311, 0.01249779, 0.01227702, 0.01295922, 0.01208275,
       0.01223631, 0.01203221, 0.01184116, 0.01146337, 0.01185

#### SG BCGD

In [ ]:
x_init = np.zeros((d,k))
step_sices = block_step_size(A)
oracle = CachedMultiLogisticRegressionOracle(predictors=A, predictions=b, n_categories = k)
x_final= gs_bcgd(x_init=x_init, oracle=oracle, block_step_size=step_sices, debug_logs=False, max_iter=5000)

#### Randomized BCGD

In [ ]:
x_init = np.zeros((d,k))
step_sices = block_step_size(A)
oracle = MultiLogisticRegressionOracle(predictors=A, predictions=b, n_categories = k)
x_final,_ = randomized_bcgd(x_init=x_init, oracle=oracle, block_step_size=step_sices, debug_logs=False, max_iter=5000)